# Telco customer churn: full Pipeline, cross-validation and error analysis

## Goal
Put preprocessing and model into one scikit-learn Pipeline, confirm it reproduces the results of [04-validation-threshold](04-validation-threshold.ipynb), check how it behaves on new input, cross-validate it, and look at validation errors by contract type. The last section exports the unfitted recipe that the final evaluation uses.

## Setup
Load the raw data and reproduce the same splits. No model is trained in this cell, and the original test set is not used for any choice in this notebook.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.base import clone
import joblib

# Project folder = the nearest parent that contains reports/final-plan.json
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reports" / "final-plan.json").exists())

data_path = PROJECT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Telco data not found: {data_path}")
df = pd.read_csv(data_path)
customer_ids = df["customerID"]
y = df["Churn"]
x = df.drop(columns=["Churn", "customerID"]).copy()
# Text to numeric is a fixed conversion; it learns no statistic from the data
x["TotalCharges"] = pd.to_numeric(x["TotalCharges"], errors="coerce")
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=2002, stratify=y
)
x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=2002, stratify=y_train
)
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["SeniorCitizen"] + x_fit.select_dtypes(exclude="number").columns.tolist()
print("fit / val / test:", len(x_fit), len(x_val), len(x_test))
assert x_fit.index.intersection(x_val.index).empty
assert x_fit.index.equals(y_fit.index)
assert x_val.index.equals(y_val.index)

fit / val / test: 4507 1127 1409


## 1. Both preprocessing routes in a ColumnTransformer
Numeric columns: median imputation, then scaling. Categorical columns: `OneHotEncoder(handle_unknown="ignore")`. Both routes go into `preprocessor`, followed by `LogisticRegression(max_iter=2800)`, forming `full_pipeline`. This cell only assembles; nothing is fitted yet.

In [2]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

full_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2800)),
])

print(full_pipeline)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['SeniorCitizen', 'gender',
                                                   'Partner', 'Dependents',
        

## 2. Fit on the fitting subset, apply to the validation set
Only `fit(x_fit, y_fit)`. The `Yes` probability column is located through `classes_`, not assumed to be column 0 or 1.

Checks: 1,127 validation predictions, probabilities within 0 to 1, every feature routed exactly once. The confusion matrix and AUC should match notebook 04 on the same split.

In [3]:
full_pipeline.fit(x_fit, y_fit)

model = full_pipeline.named_steps["model"]
yes_col = list(model.classes_).index("Yes")

val_yes_prob = full_pipeline.predict_proba(x_val)[:, yes_col]
val_pred = full_pipeline.predict(x_val)

# 1. Every validation customer gets a prediction
assert len(val_pred) == len(x_val) == 1127
assert len(val_yes_prob) == len(x_val)

# 2. Probabilities lie between 0 and 1
assert np.all((val_yes_prob >= 0) & (val_yes_prob <= 1))

# 3. Every feature enters exactly one preprocessing route
used_features = numeric_features + categorical_features
assert len(used_features) == len(set(used_features))
assert set(used_features) == set(x_fit.columns)

print("Class order:", model.classes_)
print("Confusion matrix:\n", confusion_matrix(y_val, val_pred, labels=["No", "Yes"]))
print("Validation AUC:", roc_auc_score(y_val == "Yes", val_yes_prob))
print(classification_report(y_val, val_pred))

Class order: ['No' 'Yes']
Confusion matrix:
 [[765  63]
 [131 168]]
Validation AUC: 0.8634316481669978
              precision    recall  f1-score   support

          No       0.85      0.92      0.89       828
         Yes       0.73      0.56      0.63       299

    accuracy                           0.83      1127
   macro avg       0.79      0.74      0.76      1127
weighted avg       0.82      0.83      0.82      1127



## 3. Behaviour on new customers
`probe` holds two validation customers; the first one's `Contract` is replaced by a category never seen in training. The pipeline still returns probabilities, because unknown categories are ignored by the encoder. Returning a number does not make the estimate for an unfamiliar category reliable.

Reversing the column order must not change the probabilities, since columns are selected by name. A missing required column must raise a clear error.

In [4]:
probe = x_val.head(2).copy()
probe.loc[probe.index[0], "Contract"] = "Unseen-contract-type"

prob_original = full_pipeline.predict_proba(probe)
prob_reordered = full_pipeline.predict_proba(probe[probe.columns[::-1]])

print("Probabilities with an unseen category:\n", prob_original)
print("Same probabilities after reordering columns:",
      np.allclose(prob_original, prob_reordered))

Probabilities with an unseen category:
 [[0.60090825 0.39909175]
 [0.98656178 0.01343822]]
Same probabilities after reordering columns: True


In [5]:
# Dropping a required column is expected to fail; the error is caught and shown
probe_missing = probe.drop(columns=["Contract"])
try:
    full_pipeline.predict_proba(probe_missing)
except ValueError as error:
    print("ValueError:", error)

ValueError: columns are missing: {'Contract'}


## 4. Three-fold stratified cross-validation
The whole Pipeline is passed to `cross_validate` with `x_train / y_train`, so preprocessing is re-learned inside each fold. Splitting an array that was preprocessed on all rows would leak information across folds.

The score is ROC-AUC, with one fixed configuration and no tuning. It is not an independent check of the 0.409 threshold, which was chosen on the validation set.

In [6]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2002)

cv_results = cross_validate(
    full_pipeline,
    x_train,
    y_train,
    scoring="roc_auc",
    cv=cv,
)

print("AUC per fold:", cv_results["test_score"])
print("Mean AUC:", cv_results["test_score"].mean())
print("Standard deviation:", cv_results["test_score"].std())

for fold, (fit_idx, check_idx) in enumerate(cv.split(x_train, y_train), start=1):
    assert set(fit_idx).isdisjoint(check_idx)
    assert len(fit_idx) + len(check_idx) == len(x_train)

    fit_yes_rate = y_train.iloc[fit_idx].eq("Yes").mean()
    check_yes_rate = y_train.iloc[check_idx].eq("Yes").mean()

    print(f"Fold {fold}: fit {len(fit_idx)}, check {len(check_idx)}; "
          f"churn share {fit_yes_rate:.3f} / {check_yes_rate:.3f}")

AUC per fold: [0.84192858 0.84625458 0.85236172]
Mean AUC: 0.846848296239821
Standard deviation: 0.004279950147119541
Fold 1: fit 3756, check 1878; churn share 0.265 / 0.265
Fold 2: fit 3756, check 1878; churn share 0.265 / 0.265
Fold 3: fit 3756, check 1878; churn share 0.265 / 0.266


## 5. Validation errors by contract type
The candidate threshold 0.409 is applied to the validation set only. `val_cases` has one row per customer: customerID, Contract, actual, probability, predicted. Customer IDs are aligned through `customer_ids.loc[x_val.index]`, never by position.

For each contract type: group size, TP / FP / FN / TN, actual and predicted churners, precision and recall. A zero denominator gives a missing value rather than 0. A group can show more errors simply because it is larger.

In [7]:
# 1. Apply the candidate threshold and build the per-customer table
analysis_threshold = 0.409
predicted = np.where(val_yes_prob >= analysis_threshold, "Yes", "No")

val_cases = pd.DataFrame({
    "customerID": customer_ids.loc[x_val.index],
    "Contract": x_val["Contract"],
    "actual": y_val,
    "probability": val_yes_prob,
    "predicted": predicted,
}, index=x_val.index)

assert len(val_cases) == 1127
display(val_cases.head())

,customerID,Contract,actual,probability,predicted
3132,2468-SJFLM,Month-to-month,No,0.536016,Yes
429,6152-ONASV,One year,No,0.013438,No
6200,7328-ZJAJO,Two year,No,0.001856,No
6505,7018-FPXHH,Two year,No,0.010107,No
2774,9375-MHRRS,Month-to-month,No,0.259345,No


In [8]:
# 2. Flag each customer as TP / FP / FN / TN and count the cells
val_cases["TP"] = (val_cases["actual"] == "Yes") & (val_cases["predicted"] == "Yes")
val_cases["FP"] = (val_cases["actual"] == "No")  & (val_cases["predicted"] == "Yes")
val_cases["FN"] = (val_cases["actual"] == "Yes") & (val_cases["predicted"] == "No")
val_cases["TN"] = (val_cases["actual"] == "No")  & (val_cases["predicted"] == "No")

print(val_cases[["TP", "FP", "FN", "TN"]].sum())

TP    202
FP     98
FN     97
TN    730
dtype: int64


In [9]:
# 3. Group by Contract
segment_metrics = val_cases.groupby("Contract").agg(
    group_size=("customerID", "size"),
    TP=("TP", "sum"),
    FP=("FP", "sum"),
    FN=("FN", "sum"),
    TN=("TN", "sum"),
)

# 4. The two denominators
segment_metrics["actual_yes"] = segment_metrics["TP"] + segment_metrics["FN"]
segment_metrics["predicted_yes"] = segment_metrics["TP"] + segment_metrics["FP"]

# 5. Precision and recall per group; NaN when the denominator is 0
segment_metrics["precision"] = (
    segment_metrics["TP"]
    / segment_metrics["predicted_yes"].replace(0, np.nan)
)
segment_metrics["recall"] = (
    segment_metrics["TP"]
    / segment_metrics["actual_yes"].replace(0, np.nan)
)

display(segment_metrics)

,group_size,TP,FP,FN,TN,actual_yes,predicted_yes,precision,recall
Contract,,,,,,,,,
Month-to-month,598,201,96,67,234,268,297,0.676768,0.750000
One year,249,1,2,21,225,22,3,0.333333,0.045455
Two year,280,0,0,9,271,9,0,NaN,0.000000


In [10]:
# Same breakdown for the capacity-limited alternative, threshold 0.469
val_cases_250 = val_cases[
    ["customerID", "Contract", "actual", "probability"]
].copy()

val_cases_250["predicted"] = np.where(
    val_cases_250["probability"] >= 0.469, "Yes", "No"
)

val_cases_250["TP"] = (val_cases_250["actual"] == "Yes") & (val_cases_250["predicted"] == "Yes")
val_cases_250["FP"] = (val_cases_250["actual"] == "No")  & (val_cases_250["predicted"] == "Yes")
val_cases_250["FN"] = (val_cases_250["actual"] == "Yes") & (val_cases_250["predicted"] == "No")
val_cases_250["TN"] = (val_cases_250["actual"] == "No")  & (val_cases_250["predicted"] == "No")

segment_250 = val_cases_250.groupby("Contract").agg(
    group_size=("customerID", "size"),
    TP=("TP", "sum"),
    FP=("FP", "sum"),
    FN=("FN", "sum"),
    TN=("TN", "sum"),
)

segment_250["actual_yes"] = segment_250["TP"] + segment_250["FN"]
segment_250["predicted_yes"] = segment_250["TP"] + segment_250["FP"]
segment_250["precision"] = (
    segment_250["TP"] / segment_250["predicted_yes"].replace(0, np.nan)
)
segment_250["recall"] = (
    segment_250["TP"] / segment_250["actual_yes"].replace(0, np.nan)
)

assert segment_250["group_size"].sum() == 1127
assert segment_250["predicted_yes"].sum() <= 250
display(segment_250)

,group_size,TP,FP,FN,TN,actual_yes,predicted_yes,precision,recall
Contract,,,,,,,,,
Month-to-month,598,175,73,93,257,268,248,0.705645,0.652985
One year,249,1,1,21,226,22,2,0.500000,0.045455
Two year,280,0,0,9,271,9,0,NaN,0.000000


### Observations and threshold choice

Based on the 1,127 validation customers only. Nothing is refitted here and the test set plays no part in the choice.

| Candidate threshold | Predicted Yes / contacts | TP | FP | FN | TN | Validation F1 |
|---:|---:|---:|---:|---:|---:|---:|
| 0.409 | 300 | 202 | 98 | 97 | 730 | 0.674457 |
| 0.469 | 250 | 176 | 74 | 123 | 754 | 0.641166 |

At 0.409, the month-to-month group has 268 actual churners and 201 are found (recall 75%). The one-year group has 22 churners and 1 is found (about 4.5%). The two-year group has 9 churners and none is found (0%); nobody in that group is predicted `Yes`, so its precision has a zero denominator and is reported as missing, not as 0.

With a cap of 250 contacts, 0.469 contacts 50 fewer customers than 0.409: 26 fewer true positives and 24 fewer false positives, and recall in the month-to-month group falls from 75% to about 65.3%. That is a trade-off between capacity and missed churners, not an overall improvement of the model.

Decision: **0.409** is kept as the threshold when no contact cap is confirmed; 0.469 remains an alternative for a 250-contact cap and is not selected. The groups differ a lot in size, and the two-year group has only 9 churners, so these figures do not support stable group-level or causal conclusions.

## 6. Pipeline review: three mistakes to avoid
The snippets below are deliberately wrong and are not executed.

```python
# A
full_pipeline.fit(x_val, y_val)
# B
p_yes = full_pipeline.predict_proba(x_val)[:, 0]
# C
actual_sorted = y_val.sort_values()
case_table = pd.DataFrame({"actual": actual_sorted.to_numpy(), "prob": val_yes_prob})
```

- **A** trains on the validation answers and overwrites the fitted model. Fit on `x_fit, y_fit` only; the validation set is used for `predict` / `predict_proba` and then compared with `y_val`.
- **B** takes the probability of class 0. With `classes_` equal to `['No', 'Yes']`, column 0 is not `Yes`. Locate the column with `list(model.classes_).index("Yes")`.
- **C** `sort_values()` reorders `y_val` and `to_numpy()` drops the index, so labels and probabilities no longer belong to the same customer. Build the table on the shared index instead: `pd.DataFrame({"actual": y_val, "prob": pd.Series(val_yes_prob, index=x_val.index)})`.

Two further checks: `customerID` and `Churn` must not be among the model inputs, and test results must never be used to pick a model or a threshold.

In [11]:
assert {"customerID", "Churn"}.isdisjoint(x_fit.columns)
assert x_val.index.equals(y_val.index)
assert len(val_yes_prob) == len(x_val) == 1127
assert "Yes" in full_pipeline.classes_
print("Input and alignment checks passed.")

Input and alignment checks passed.


## 7. Recipe handed to the final evaluation
What is exported is a `clone` of the pipeline: the unfitted preprocessing and model configuration, carrying no training data. The final evaluation fits it again on the full original training set.

The export has already been done and the recipe's SHA-256 is recorded in `reports/final-plan.json`, together with the threshold and the reason for choosing it. The switch stays off so that re-running this notebook cannot overwrite the frozen recipe.

In [12]:
EXPORT_RECIPE = False  # already exported and frozen
if EXPORT_RECIPE:
    artifact_dir = PROJECT / "artifacts"
    artifact_dir.mkdir(exist_ok=True)
    joblib.dump(clone(full_pipeline), artifact_dir / "pipeline-recipe.joblib")
    print("Unfitted recipe exported.")
else:
    print("Recipe export skipped: artifacts/pipeline-recipe.joblib is already frozen.")

Recipe export skipped: artifacts/pipeline-recipe.joblib is already frozen.


## Summary
- **Pipeline.** Both preprocessing routes and the model are assembled and fitted in one object. On the same validation set the confusion matrix is TN 765, FP 63, FN 131, TP 168, churn-class F1 about 0.63 at the default 0.5 threshold and AUC about 0.8634, identical to notebook 04.
- **Input checks.** An unseen category, a reversed column order and a missing column behave as intended.
- **Cross-validation.** Three-fold ROC-AUC on the original training set: 0.8419, 0.8463, 0.8524 (mean 0.8468, standard deviation 0.0043). The folds do not overlap, each has 3,756 fitting and 1,878 checking customers, and the churn share is about 0.265 on both sides.
- **Configuration.** Numeric columns `tenure`, `MonthlyCharges`, `TotalCharges`: median imputation, then `StandardScaler`. Categorical columns: `OneHotEncoder(handle_unknown="ignore")`. Model: `LogisticRegression(max_iter=2800)`.
- **Threshold.** 0.409 is frozen as the threshold without a contact cap; 0.469 is kept only as the unselected 250-contact alternative.
- **Limit.** The original test set was looked at once in the first baseline notebook. It was not used here to choose the model or the threshold.